In [1]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import GridSearchCV, LeaveOneOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, recall_score, precision_score, f1_score, average_precision_score, precision_recall_fscore_support, roc_auc_score
import math
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
# jupyter nbconvert --to script DataProcessing_run.ipynb

___

- Regression

In [2]:
reg_param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5],
    'min_samples_leaf': [1, 2]}
reg_param_grid_xgb = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2]}
reg_param_grid_svr = {
    'C': [1], # [1, 10]
    'kernel': ['rbf']}
reg_param_grid_mlp = {
    'hidden_layer_sizes': [(100,), (50, 50)], 
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'learning_rate': ['constant', 'adaptive']}

In [3]:
def reg_process_leave_one_out(patient, df, patient_column, target_column, model, param_grid):
    train_df = df[df[patient_column] != patient]
    test_df = df[df[patient_column] == patient]
    X_train = train_df.drop(columns=['Session', 'Segment', target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=['Session', 'Segment', target_column])
    y_test = test_df[target_column]
    scaler = StandardScaler() 
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='neg_mean_squared_error',
        cv=2,  # 3
        verbose=0,
        n_jobs=-1)
    if isinstance(model, SVR):
        grid_search.fit(X_train_scaled, y_train)
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test_scaled)
    else:
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
    return {
        "Model": type(model).__name__,
        "Patient_ID": patient,
        "Best_Params": grid_search.best_params_,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "y_true": y_test.values.tolist(),   
        "y_pred": y_pred.tolist(),
        "X_test": X_test.values.tolist(), 
        "model_fitted": best_model}

- Classification

In [4]:
class_param_grid_rf = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced']}
class_param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 1],
    'colsample_bytree': [0.7, 1]}
class_param_grid_logreg = {
    'C': [0.01, 0.1, 1, 10, 100],         # 'C': [0.1, 1]
    'penalty': ['l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced']}
class_param_grid_mlp = {
    'hidden_layer_sizes': [(100,), (50, 50), (100, 50)],
    'activation': ['relu'],
    'solver': ['adam'],
    'learning_rate': ['constant', 'adaptive'],
    'alpha': [0.0001, 0.001],  # Regularization
    'early_stopping': [True]}

In [5]:
def class_process_leave_one_out(patient, df, patient_column, target_column, model, param_grid):
    train_df = df[df[patient_column] != patient]
    test_df  = df[df[patient_column] == patient]
    X_train = train_df.drop(columns=['Session', 'Segment', target_column])
    y_train = train_df[target_column].values
    X_test  = test_df.drop(columns=['Session', 'Segment', target_column])
    y_test  = test_df[target_column].values
    unique_train_classes = sorted(np.unique(y_train))
    if len(unique_train_classes) < 2:
        return None  
    labels_all = sorted(set(np.unique(y_train)) | set(np.unique(y_test)))
    is_multiclass = len(labels_all) > 2
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    try:
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            scoring='f1_macro',
            cv=3,
            verbose=0,
            n_jobs=-1)
        grid_search.fit(X_train_scaled, y_train)
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test_scaled)
        if is_multiclass:
            p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
                y_test, y_pred, labels=labels_all, average="macro", zero_division=0)
            out = {
                "Model": type(best_model).__name__,
                "Patient_ID": patient,
                "Best_Params": grid_search.best_params_,
                "Recall":   float(r_macro),      # Macro recall (M)
                "F1_Score": float(f1_macro),     # Macro F1
                "Precision": float(p_macro),     # Macro precision
                "y_true":  y_test.tolist(),
                "y_pred":  y_pred.tolist()}
        else:
            bin_labels = [0, 1] if set(labels_all) == {0, 1} else labels_all
            rec_macro = recall_score(y_test, y_pred, labels=bin_labels, average='macro', zero_division=0)
            if set(bin_labels) == {0, 1}:
                rec_pos = recall_score(y_test, y_pred, average='binary', pos_label=1, zero_division=0)
                prec    = precision_score(y_test, y_pred, zero_division=0, pos_label=1)
                f1      = f1_score(y_test, y_pred, zero_division=0, pos_label=1)
            else:
                # robust fallback when labels are not {0,1}
                # use the "largest" class as positive by convention
                pos_lab = max(bin_labels)
                rec_pos = recall_score(y_test, y_pred, average='binary', pos_label=pos_lab, zero_division=0)
                prec    = precision_score(y_test, y_pred, zero_division=0, pos_label=pos_lab)
                f1      = f1_score(y_test, y_pred, zero_division=0, pos_label=pos_lab)
            out = {
                "Model": type(best_model).__name__,
                "Patient_ID": patient,
                "Best_Params": grid_search.best_params_,
                "Recall":     float(rec_macro),  # Macro-per-fold (M)
                "Recall_Pos": float(rec_pos),    # Positive-class (P)
                "F1_Score":   float(f1),         # P
                "Precision":  float(prec),       # P
                "y_true":  y_test.tolist(),
                "y_pred":  y_pred.tolist()}
        return out
    except ValueError:
        return None